# RBDisco API Example
RBDisco is a plugin to ActiTect to predict RBD status from multi-night actigraphy recordings. It hosts the pretrained models as described in the main [README](../../README.md)/paper. We provide batch processing via the CLI, but you can also use `actitect.rbdisco.predict()` for more flexibility.

In [1]:

from pathlib import Path
file = Path.cwd().parent.joinpath('actitect/example.cwa')

from actitect.rbdisco import predict
from actitect.api import process, compute_sleep_motor_features

SUBJECT_ID = 'ID-1'

processed_df, _ = process(file, subject_id=SUBJECT_ID) # use actitect to load and preprocess the data
feat_df = compute_sleep_motor_features(processed_df, subject_id=SUBJECT_ID)

night_level = predict(
    data=feat_df,  # we use the extracted sleep-motor features for prediction; you can also directly pass a pathlike object pointing to a valid actigraphy binary (.cwa, .bin, .gt3x or .csv) and it will run the features extraction under the hood
	return_level='night',  # this will return nightly scores and predictions
    subject_id=SUBJECT_ID  # optional
)

print(night_level)

/Users/david/anaconda3/envs/RBDisco/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 (18:16:14) - [actitect.actimeter.factory|INFO]: (io: ID-1) detected 'AxivityAx6' device.
 (18:16:14) - [actitect.actimeter.basedevice|INFO]: (io: ID-1) parsing data from 'example.cwa'.
 (18:16:30) - [actitect.utils.data_utils|INFO]: (io: ID-1) found 38 duplicate timestamps.removing ~0.38s of data
 (18:16:32) - [actitect.actimeter.basedevice|INFO]: (io: ID-1) successfully loaded raw data. (17.75s)
 (18:16:32) - [actitect.actimeter.basedevice|INFO]: (resampling: ID-1) non-uniform sampling rate in raw data detected: fs = 100.05 ± 24.85 Hz
 (18:17:22) - [actitect.actimeter.basedevice|INFO]: (resampling: ID-1) successful. (50.70s)
 (18:17:26) - [actitect.actimeter.basedevice|INFO]: (filter: ID-1) successful. (3.85s)
 (18:17:33) - [actitect.actimeter.basedevice|INFO]: (calibration: ID-1) calibration error summary: before=18.35mg after=2.71mg.(62 iterations)
 (18:17:33) - [actitect.actimeter.basedevice|INFO]: (calibration: ID-1) successful. (6.72s)
 (18:17:37) - [actitect.actimeter.basedevic

[PROGRESS]: Computing sleep features: 100%|■■■■■■■■■■| 3/3 [14:30<00:00, 290.14s/it, sample=ID-1, night=3/3, features=99.8%]                                


 (18:32:56) - [actitect.utils.data_utils|INFO]: found 65688 problematic cells (5e+01% of 92 cols×1571 rows) in 'local_feat_df': NaN: 65688. Using drop=True, replace=None.
 (18:32:57) - [actitect.utils.data_utils|INFO]: local to global aggregation done. (0.17s)
 (18:32:57) - [actitect.rbdisco.core.types|INFO]: applying pre-fitted global robust scaler.
  subject  night_id  proba_rbd  pred
0    ID-1         0   0.419406     0
1    ID-1         1   0.381201     0
2    ID-1         2   0.375624     0


Here, each row corresponds to one night of the recording, assigned to a specific RBD probability score in [0,1] and a thresholded binary prediction. To account for night-to-night variability and increase prediction robustness, we aggregate the per-night predictions to patient level (details in paper):

In [2]:
patient_level = predict(
    data=feat_df,  # again, you could also simply pass 'file' here to automatically compute the features, but here we want to use the cached features
	return_level='patient',  # this is the default: nightly scores are aggregated to patient level
    subject_id=SUBJECT_ID  # optional
)

print(patient_level)

 (18:32:57) - [actitect.rbdisco.core.types|INFO]: applying pre-fitted global robust scaler.
     id  n_total_nights  mean_prob_per_night  pred(ensemble_major)
0  ID-1               3             0.392077                     0


By default, `.predict()` will use the pretrained multi-center as an estimator. You can also use the single-center model (refer to paper) by specifying:
```
patient_level = predict(
    data=file,
	return_level='patient',
	model='singleCenterCore',  # additional arg to use the single-center model instead
    subject_id='ID-1'  # optional
)
```